In [ ]:
# ⚠️ Please make sure you have adequate GPU memory.
# ⚠️ Please make sure your EnergyPlus version is 9.6 for successful running.

! pip install -U bitsandbytes -q # pip this repo at the first time.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.2 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from peft import PeftModel, PeftConfig
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the EPlus-LLMv2 config.
peft_model_id = "EPlus-LLM/EPlus-LLMv2"
config = PeftConfig.from_pretrained(peft_model_id)

# Load the base LLM, flan-t5-xxl, and tokenizer
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-xxl",
    quantization_config=quant_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-xxl")

# Load the Lora model
model = PeftModel.from_pretrained(model, peft_model_id)

# Generation config
generation_config = model.generation_config
generation_config.max_new_tokens = 5000
generation_config.temperature = 0.1
generation_config.top_p = 0.1
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

# Please provide your input here — a description of the desired building
# For more details, please refer to the paper: https://doi.org/10.1016/j.autcon.2025.106223
input=f"""
Simulate a U-shaped building that is 99.73 meters high, with a gable roof.
The horizontal segment is 732.31 meters long and 17.54 meters wide. The left vertical segment is 256.31 meters long and 206.96 meters wide. The right vertical segment is 431.54 meters long and 62 meters wide.
The roof ridge is 8.77 meters to the length side of the horizontal segment, and 128.16 meters, 215.77 meters to the width side of the vertical segments, respectively.
The attic height is 139.71 meters. The building orientation is 62 degrees to the north. The building has 3 thermal zones with each segment as one thermal zone.
The window-to-wall ratio is 0.32. The window sill height is 33.91 meters, the window height is 65.82 meters, and the window jamb width is 0.01 meters.
The window U-factor is 6.36 W/m2K and the SHGC is 0.89.
The wall is made of wood, with a thickness of 0.48 meters and the wall insulation is RSI 1.6 m2K/W, U-factor 0.63 W/m2K.
The roof is made of metal, with a thickness of 0.09 meters and the roof insulation is RSI 5.4 m2K/W, U-factor 0.19 W/m2K.
The floor is made of concrete, covered with carpet. The ventilation rate is 2.32 ach. The infiltration rate is 0.55 ach.
The people density is 16.61 m2/person, the light density is 4.48 W/m2, and the electric equipment density is 22.63 W/m2.
Occupancy starts at 7:00 and ends at 18:00. The occupancy rate is 1. The unoccupancy rate is 0.3.
The heating setpoint is 21.54 Celsius in occupancy period and 15.86 Celsius in unoccupancy period.
The cooling setpoint is 22.6 Celsius in occupancy period and 26.72 Celsius in unoccupancy period.
"""

# EPlus-LLM generating...
input_ids = tokenizer(input, return_tensors="pt", truncation=False).to(device)
generated_ids = model.generate(input_ids = input_ids.input_ids,
                           attention_mask = input_ids.attention_mask,
                           generation_config = generation_config)
generated_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Default thermal zones setting
zone_1 = """ZoneHVAC:EquipmentConnections,Thermal Zone 1,Thermal Zone 1 Equipment,Thermal Zone 1 Ideal Loads Supply Inlet,,Thermal Zone 1 Zone Air Node,Thermal Zone 1 Return Outlet;
ZoneHVAC:EquipmentList,Thermal Zone 1 Equipment,SequentialLoad,ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 1 Ideal Loads Air System,1,1,,;
ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 1 Ideal Loads Air System,,Thermal Zone 1 Ideal Loads Supply Inlet,,,50,13,0.0156,0.0077,NoLimit,,,NoLimit,,,,,ConstantSensibleHeatRatio,0.7,None,,,None,NoEconomizer,None,0.7,0.65;
ZoneControl:Thermostat,Thermal Zone 1 Thermostat,Thermal Zone 1,Thermostat Schedule,ThermostatSetpoint:DualSetpoint,Thermostat Setpoint Dual Setpoint,,,,,,,0;
Sizing:Zone,Thermal Zone 1,SupplyAirTemperature,14,11.11,SupplyAirTemperature,40,11.11,0.0085,0.008,Ventilation,,,DesignDay,0,0.000762,0,0,DesignDay,0,0.002032,0.1415762,0.3,,No;"""
zone_2 = """ZoneHVAC:EquipmentConnections,Thermal Zone 2,Thermal Zone 2 Equipment,Thermal Zone 2 Ideal Loads Supply Inlet,,Thermal Zone 2 Zone Air Node,Thermal Zone 2 Return Outlet;
ZoneHVAC:EquipmentList,Thermal Zone 2 Equipment,SequentialLoad,ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 2 Ideal Loads Air System,1,1,,;
ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 2 Ideal Loads Air System,,Thermal Zone 2 Ideal Loads Supply Inlet,,,50,13,0.0156,0.0077,NoLimit,,,NoLimit,,,,,ConstantSensibleHeatRatio,0.7,None,,,None,NoEconomizer,None,0.7,0.65;
ZoneControl:Thermostat,Thermal Zone 2 Thermostat,Thermal Zone 2,Thermostat Schedule,ThermostatSetpoint:DualSetpoint,Thermostat Setpoint Dual Setpoint,,,,,,,0;
Sizing:Zone,Thermal Zone 2,SupplyAirTemperature,14,11.11,SupplyAirTemperature,40,11.11,0.0085,0.008,Ventilation,,,DesignDay,0,0.000762,0,0,DesignDay,0,0.002032,0.1415762,0.3,,No;"""
zone_3 = """ZoneHVAC:EquipmentConnections,Thermal Zone 3,Thermal Zone 3 Equipment,Thermal Zone 3 Ideal Loads Supply Inlet,,Thermal Zone 3 Zone Air Node,Thermal Zone 3 Return Outlet;
ZoneHVAC:EquipmentList,Thermal Zone 3 Equipment,SequentialLoad,ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 3 Ideal Loads Air System,1,1,,;
ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 3 Ideal Loads Air System,,Thermal Zone 3 Ideal Loads Supply Inlet,,,50,13,0.0156,0.0077,NoLimit,,,NoLimit,,,,,ConstantSensibleHeatRatio,0.7,None,,,None,NoEconomizer,None,0.7,0.65;
ZoneControl:Thermostat,Thermal Zone 3 Thermostat,Thermal Zone 3,Thermostat Schedule,ThermostatSetpoint:DualSetpoint,Thermostat Setpoint Dual Setpoint,,,,,,,0;
Sizing:Zone,Thermal Zone 3,SupplyAirTemperature,14,11.11,SupplyAirTemperature,40,11.11,0.0085,0.008,Ventilation,,,DesignDay,0,0.000762,0,0,DesignDay,0,0.002032,0.1415762,0.3,,No;"""
zone_4 = """ZoneHVAC:EquipmentConnections,Thermal Zone 4,Thermal Zone 4 Equipment,Thermal Zone 4 Ideal Loads Supply Inlet,,Thermal Zone 4 Zone Air Node,Thermal Zone 4 Return Outlet;
ZoneHVAC:EquipmentList,Thermal Zone 4 Equipment,SequentialLoad,ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 4 Ideal Loads Air System,1,1,,;
ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 4 Ideal Loads Air System,,Thermal Zone 4 Ideal Loads Supply Inlet,,,50,13,0.0156,0.0077,NoLimit,,,NoLimit,,,,,ConstantSensibleHeatRatio,0.7,None,,,None,NoEconomizer,None,0.7,0.65;
ZoneControl:Thermostat,Thermal Zone 4 Thermostat,Thermal Zone 4,Thermostat Schedule,ThermostatSetpoint:DualSetpoint,Thermostat Setpoint Dual Setpoint,,,,,,,0;
Sizing:Zone,Thermal Zone 4,SupplyAirTemperature,14,11.11,SupplyAirTemperature,40,11.11,0.0085,0.008,Ventilation,,,DesignDay,0,0.000762,0,0,DesignDay,0,0.002032,0.1415762,0.3,,No;"""
zone_5 = """ZoneHVAC:EquipmentConnections,Thermal Zone 5,Thermal Zone 5 Equipment,Thermal Zone 5 Ideal Loads Supply Inlet,,Thermal Zone 5 Zone Air Node,Thermal Zone 5 Return Outlet;
ZoneHVAC:EquipmentList,Thermal Zone 5 Equipment,SequentialLoad,ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 5 Ideal Loads Air System,1,1,,;
ZoneHVAC:IdealLoadsAirSystem,Thermal Zone 5 Ideal Loads Air System,,Thermal Zone 5 Ideal Loads Supply Inlet,,,50,13,0.0156,0.0077,NoLimit,,,NoLimit,,,,,ConstantSensibleHeatRatio,0.7,None,,,None,NoEconomizer,None,0.7,0.65;
ZoneControl:Thermostat,Thermal Zone 5 Thermostat,Thermal Zone 5,Thermostat Schedule,ThermostatSetpoint:DualSetpoint,Thermostat Setpoint Dual Setpoint,,,,,,,0;
Sizing:Zone,Thermal Zone 5,SupplyAirTemperature,14,11.11,SupplyAirTemperature,40,11.11,0.0085,0.008,Ventilation,,,DesignDay,0,0.000762,0,0,DesignDay,0,0.002032,0.1415762,0.3,,No;"""
generated_output = generated_output.replace(";",";\n")
generated_output = generated_output.replace("Ideal Load System Setting for Thermal Zone 1;", zone_1)
generated_output = generated_output.replace("Ideal Load System Setting for Thermal Zone 2;", zone_2)
generated_output = generated_output.replace("Ideal Load System Setting for Thermal Zone 3;", zone_3)
generated_output = generated_output.replace("Ideal Load System Setting for Thermal Zone 4;", zone_4)
generated_output = generated_output.replace("Ideal Load System Setting for Thermal Zone 5;", zone_5)

# Load the rest port of IDF file.
file_path = "v2_nextpart.idf"
output_path = "v2_final.idf"
# Output the building energy model in IDF file
with open(file_path, 'r', encoding='utf-8') as file:
    nextpart = file.read()
final_text = nextpart + "\n\n" + generated_output
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(final_text)
print(f"Building Energy Model Auto-Generated: {output_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


adapter_config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

TypeError: T5ForConditionalGeneration.__init__() got an unexpected keyword argument 'load_in_8bit'